# Delta Lake Basics — 10: Table Maintenance

## What you will learn
A Delta table needs regular maintenance to stay performant.
Skipping maintenance leads to slow queries, wasted storage, and bloated metadata.

In this notebook:
1. VACUUM — safely removing old physical files
2. OPTIMIZE — compaction scheduling strategy
3. History management — controlling retention
4. DESCRIBE DETAIL — monitoring table health
5. Table properties — configuring auto-maintenance
6. Complete maintenance runbook for production


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/delta_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('delta-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Delta Lake ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 20:21:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Delta Lake ready


26/04/10 20:21:29 WARN TaskSetManager: Stage 0 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:21:31 WARN TaskSetManager: Stage 1 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.


Dataset ready: 100,000 rows


In [2]:
TABLE = f"{DATA_DIR}/10_maintenance"
shutil.rmtree(TABLE, ignore_errors=True)

# Create a table with maintenance needs:
# many appends, updates, deletes → lots of old files
df.write.format("delta").mode("overwrite").save(TABLE)

for i in range(15):
    df.limit(2000).write.format("delta").mode("append").save(TABLE)

spark.sql(f"UPDATE delta.`{TABLE}` SET status='shipped' WHERE status='confirmed' AND order_id%3=0")
spark.sql(f"DELETE FROM delta.`{TABLE}` WHERE status='cancelled' AND price < 50")
spark.sql(f"UPDATE delta.`{TABLE}` SET revenue=revenue*1.1 WHERE country='US'")

print("Table after many operations:")
DeltaTable.forPath(spark, TABLE).detail() \
    .select("numFiles","sizeInBytes").show()
print()

# Count physical files (including old logically-deleted ones)
all_parquet = list(pathlib.Path(TABLE).glob("*.parquet"))
print(f"Physical Parquet files on disk: {len(all_parquet)}  (includes old deleted files)")
logical = DeltaTable.forPath(spark, TABLE).detail().collect()[0]["numFiles"]
print(f"Logical files (current version): {logical}")
print(f"Stale files (VACUUM candidates) : {len(all_parquet) - logical}")

26/04/10 20:21:33 WARN TaskSetManager: Stage 4 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:21:38 WARN TaskSetManager: Stage 5 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:21:38 WARN TaskSetManager: Stage 7 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:21:39 WARN TaskSetManager: Stage 9 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:21:40 WARN TaskSetManager: Stage 11 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:21:41 WARN TaskSetManager: Stage 13 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:21:42 WARN TaskSetManager: Stage 15 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 2

Table after many operations:
+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|       3|    2357331|
+--------+-----------+


Physical Parquet files on disk: 29  (includes old deleted files)
Logical files (current version): 3
Stale files (VACUUM candidates) : 26


## Step 1 — VACUUM: Removing Old Files

VACUUM physically deletes Parquet files that are no longer referenced
by any active snapshot. These accumulate from UPDATE/DELETE/OPTIMIZE operations.

**Safety mechanism:** default retention is 7 days (168 hours).
Files younger than retention are NOT deleted — protecting concurrent readers
and time travel within the retention window.


In [3]:
# Disable safety check BEFORE VACUUM with low retention (demo only — NEVER in production!)
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

# DRY RUN first — see what would be deleted without deleting
print("VACUUM DRY RUN — files that would be removed:")
spark.sql(f"VACUUM delta.`{TABLE}` RETAIN 0 HOURS DRY RUN").show(10, truncate=False)

# Actual VACUUM
before_count = len(list(pathlib.Path(TABLE).glob("*.parquet")))
spark.sql(f"VACUUM delta.`{TABLE}` RETAIN 0 HOURS")
after_count = len(list(pathlib.Path(TABLE).glob("*.parquet")))

print(f"\nVACUUM results:")
print(f"  Physical files before: {before_count}")
print(f"  Physical files after : {after_count}")
print(f"  Files deleted        : {before_count - after_count}")

# Re-enable safety check
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")
print("\nSafety check re-enabled ✅")

# After VACUUM, time travel before retention window is impossible
print()
print("⚠️  After VACUUM RETAIN 0 HOURS — time travel to old versions fails:")
try:
    spark.read.format("delta").option("versionAsOf", 1).load(TABLE).count()
    print("  Still readable (within retention)")
except Exception as e:
    print(f"  ❌ Version 1 no longer accessible (files deleted)")

VACUUM DRY RUN — files that would be removed:


+--------------------------------------------------------------------------------------------------------------------+
|path                                                                                                                |
+--------------------------------------------------------------------------------------------------------------------+
|file:/workspace/data/delta_basics/10_maintenance/part-00000-aa66fc91-45c5-42f1-a740-11fd58c65a76-c000.snappy.parquet|
|file:/workspace/data/delta_basics/10_maintenance/part-00000-40052853-a5a4-4150-b304-58daac7ee587-c000.snappy.parquet|
|file:/workspace/data/delta_basics/10_maintenance/part-00000-8e44b55e-a898-4da6-9ed6-24964b71ca27-c000.snappy.parquet|
|file:/workspace/data/delta_basics/10_maintenance/part-00000-bccdb866-7244-4279-b870-5f99f747265e-c000.snappy.parquet|
|file:/workspace/data/delta_basics/10_maintenance/part-00000-ccb9914b-cada-4254-8f06-2bee1d4a51b4-c000.snappy.parquet|
|file:/workspace/data/delta_basics/10_maintenanc

Deleted 26 files and directories in a total of 1 directories.

VACUUM results:
  Physical files before: 29
  Physical files after : 3
  Files deleted        : 26

Safety check re-enabled ✅

⚠️  After VACUUM RETAIN 0 HOURS — time travel to old versions fails:
  Still readable (within retention)


## Step 2 — History Management


In [4]:
# How many versions do we have?
history = DeltaTable.forPath(spark, TABLE).history()
version_count = history.count()
print(f"Total versions in history: {version_count}")
history.select("version","timestamp","operation").orderBy("version").show(10, truncate=False)

# Configure retention for history metadata
spark.sql(f"""
    ALTER TABLE delta.`{TABLE}`
    SET TBLPROPERTIES (
        'delta.logRetentionDuration'       = 'interval 30 days',
        'delta.deletedFileRetentionDuration'= 'interval 7 days'
    )
""")
print("\nRetention configured:")
print("  logRetentionDuration        : 30 days (how long history JSON kept)")
print("  deletedFileRetentionDuration: 7 days  (how long deleted Parquet kept)")
print()
print("These settings control the time travel window and VACUUM safety period.")

Total versions in history: 21
+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|0      |2026-04-10 20:21:37.221|WRITE    |
|1      |2026-04-10 20:21:38.473|WRITE    |
|2      |2026-04-10 20:21:39.232|WRITE    |
|3      |2026-04-10 20:21:39.978|WRITE    |
|4      |2026-04-10 20:21:40.727|WRITE    |
|5      |2026-04-10 20:21:41.487|WRITE    |
|6      |2026-04-10 20:21:42.266|WRITE    |
|7      |2026-04-10 20:21:43.098|WRITE    |
|8      |2026-04-10 20:21:43.932|WRITE    |
|9      |2026-04-10 20:21:44.835|WRITE    |
+-------+-----------------------+---------+
only showing top 10 rows

Retention configured:
  logRetentionDuration        : 30 days (how long history JSON kept)
  deletedFileRetentionDuration: 7 days  (how long deleted Parquet kept)

These settings control the time travel window and VACUUM safety period.


## Step 3 — DESCRIBE DETAIL: Health Monitoring


In [5]:
print("=== Table Health Dashboard ===")
print()

detail = DeltaTable.forPath(spark, TABLE).detail().collect()[0]
history = DeltaTable.forPath(spark, TABLE).history().collect()

total_rows  = spark.read.format("delta").load(TABLE).count()
num_files   = detail["numFiles"]
size_mb     = detail["sizeInBytes"] / 1024 / 1024
avg_file_mb = size_mb / num_files if num_files > 0 else 0
num_versions= len(history)

print(f"  Table path     : {TABLE}")
print(f"  Rows           : {total_rows:,}")
print(f"  Logical files  : {num_files}")
print(f"  Total size     : {size_mb:.1f} MB")
print(f"  Avg file size  : {avg_file_mb:.1f} MB")
print(f"  Versions       : {num_versions}")
print(f"  Last operation : {history[0]['operation'] if history else 'none'}")
print()

# Health signals
print("Health signals:")
if avg_file_mb < 10:
    print(f"  ⚠️  Small files ({avg_file_mb:.1f} MB avg) — run OPTIMIZE")
else:
    print(f"  ✅ File sizes ok ({avg_file_mb:.1f} MB avg)")

if num_versions > 50:
    print(f"  ⚠️  Many versions ({num_versions}) — consider VACUUM")
else:
    print(f"  ✅ Version count ok ({num_versions})")

=== Table Health Dashboard ===

  Table path     : /workspace/data/delta_basics/10_maintenance
  Rows           : 128,802
  Logical files  : 3
  Total size     : 2.2 MB
  Avg file size  : 0.7 MB
  Versions       : 22
  Last operation : SET TBLPROPERTIES

Health signals:
  ⚠️  Small files (0.7 MB avg) — run OPTIMIZE
  ✅ Version count ok (22)


## Step 4 — Complete Maintenance Runbook


In [6]:
print("""
╔══════════════════════════════════════════════════════════════╗
║         DELTA LAKE MAINTENANCE RUNBOOK                       ║
╚══════════════════════════════════════════════════════════════╝

DAILY (after peak writes):
─────────────────────────
1. OPTIMIZE recent partitions
   spark.sql(f"OPTIMIZE delta.`/path` WHERE date = current_date()")

2. Check file count health
   detail = DeltaTable.forPath(spark, path).detail().collect()[0]
   avg_mb = detail['sizeInBytes'] / detail['numFiles'] / 1024/1024
   if avg_mb < 64:  run OPTIMIZE

WEEKLY:
──────
3. OPTIMIZE entire table with ZORDER
   spark.sql("OPTIMIZE delta.`/path` ZORDER BY (category, country)")

4. VACUUM (keep 7 days for time travel)
   spark.sql("VACUUM delta.`/path` RETAIN 168 HOURS")

MONTHLY:
────────
5. Review and expire old snapshots
   spark.sql("DESCRIBE HISTORY delta.`/path`")  -- check version count

6. Review table properties and retention settings
   spark.sql("DESCRIBE DETAIL delta.`/path`")

WHEN THINGS GO WRONG:
─────────────────────
Bad write/transform? → RESTORE TABLE TO VERSION AS OF N
Schema mismatch?     → Check history, use VERSION AS OF to find last good version
Performance slow?    → Check avg file size, run OPTIMIZE ZORDER BY
Storage growing?     → Run VACUUM, check deletedFileRetentionDuration

CONFIGURATIONS:
──────────────
ALTER TABLE delta.`/path` SET TBLPROPERTIES (
    'delta.enableChangeDataFeed'          = 'true',
    'delta.enableDeletionVectors'         = 'true',
    'delta.logRetentionDuration'          = 'interval 30 days',
    'delta.deletedFileRetentionDuration'  = 'interval 7 days'
)
""")


╔══════════════════════════════════════════════════════════════╗
║         DELTA LAKE MAINTENANCE RUNBOOK                       ║
╚══════════════════════════════════════════════════════════════╝

DAILY (after peak writes):
─────────────────────────
1. OPTIMIZE recent partitions
   spark.sql(f"OPTIMIZE delta.`/path` WHERE date = current_date()")

2. Check file count health
   detail = DeltaTable.forPath(spark, path).detail().collect()[0]
   avg_mb = detail['sizeInBytes'] / detail['numFiles'] / 1024/1024
   if avg_mb < 64:  run OPTIMIZE

WEEKLY:
──────
3. OPTIMIZE entire table with ZORDER
   spark.sql("OPTIMIZE delta.`/path` ZORDER BY (category, country)")

4. VACUUM (keep 7 days for time travel)
   spark.sql("VACUUM delta.`/path` RETAIN 168 HOURS")

MONTHLY:
────────
5. Review and expire old snapshots
   spark.sql("DESCRIBE HISTORY delta.`/path`")  -- check version count

6. Review table properties and retention settings
   spark.sql("DESCRIBE DETAIL delta.`/path`")

WHEN THINGS GO WRO

## Summary: Complete Delta Basics Recap

You have now covered all 10 core Delta Lake topics:

| Notebook | Topic | Key takeaway |
|---|---|---|
| 01 | First table | Delta = Parquet + `_delta_log` |
| 02 | Read/Write | Multiple write modes, idempotent writes, replaceWhere |
| 03 | ACID | Atomicity, schema enforcement, snapshot isolation |
| 04 | DML | UPDATE/DELETE/MERGE for row-level operations |
| 05 | Time travel | VERSION AS OF, TIMESTAMP AS OF, RESTORE |
| 06 | OPTIMIZE/ZORDER | Compact small files, co-locate data for fast filters |
| 07 | Schema | Enforcement by default, mergeSchema for evolution |
| 08 | CDF | Track row-level changes for incremental pipelines |
| 09 | Partitioning | partitionBy + Liquid Clustering as modern alternative |
| 10 | Maintenance | VACUUM + OPTIMIZE + monitoring schedule |

**Continue with:** `data_formats_storage/03_delta_advanced.ipynb` for
Deletion Vectors, low-shuffle MERGE, dynamic partition overwrite, and cloning.
